<a href="https://colab.research.google.com/github/Karna14314/Hugginface_models/blob/main/Fine_tune_Sentiment_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

```markdown
# Project: Fast DistilBERT Sentiment Fine-Tuning & Deployment
This notebook demonstrates a recruiter-ready NLP pipeline: fine-tuning a transformer model, adding explainability, and preparing for Hugging Face deployment.
```

In [6]:
!pip install -q datasets transformers[torch] evaluate captum huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 17.0 MB/s eta 0:00:00


In [7]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Load a subset of IMDb (5k training, 1k test for speed)
dataset = load_dataset("imdb")
train_sub = dataset["train"].shuffle(seed=42).select(range(5000))
test_sub = dataset["test"].shuffle(seed=42).select(range(1000))

# 2. Preprocessing
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = train_sub.map(tokenize_function, batched=True)
tokenized_test = test_sub.map(tokenize_function, batched=True)

# 3. Model & Metrics
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# 4. Fast Training (1 Epoch)
training_args = TrainingArguments(
    output_dir="sentiment-distilbert-fast",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.275132,0.881000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=313, training_loss=0.3488577943259535, metrics={'train_runtime': 246.0023, 'train_samples_per_second': 20.325, 'train_steps_per_second': 1.272, 'total_flos': 662336993280000.0, 'train_loss': 0.3488577943259535, 'epoch': 1.0})

```markdown
### Explainability: Visualizing Word Importance
Using simple gradient-based saliency to show which words influenced the sentiment prediction.
```

In [12]:
def get_explainability(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    # Enable gradients for embeddings
    embeddings = model.get_input_embeddings()(input_ids)
    embeddings.retain_grad()

    outputs = model(inputs_embeds=embeddings, attention_mask=inputs["attention_mask"])
    score = torch.max(outputs.logits)
    score.backward()

    # Calculate saliency (norm of gradients)
    saliency = embeddings.grad.data.abs().sum(dim=-1).squeeze(0)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    return list(zip(tokens, saliency.tolist()))

sample_text = "This movie was absolutely fantastic and the acting was superb!"
explanation = get_explainability(sample_text)
print("Token Importance (Top 5):", sorted(explanation, key=lambda x: x[1], reverse=True)[:5])

Token Importance (Top 5): [('superb', 1.0), ('fantastic', 0.8203502893447876), ('!', 0.4638984799385071), ('absolutely', 0.4613925516605377), ('movie', 0.36282017827033997)]


```markdown
### Deploy to Hugging Face Spaces
To push to Spaces, you'll need your [Hugging Face Write Token](https://huggingface.co/settings/tokens).
```

In [14]:
from huggingface_hub import notebook_login, HfApi

# Uncomment the next line to login
notebook_login()

# Code to push model
model.push_to_hub("ncncomplete/distilbert-imdb-sentiment")
tokenizer.push_to_hub("ncncomplete/distilbert-imdb-sentiment")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...i6v7tyh/model.safetensors:   0%|          |  575kB /  268MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ncncomplete/distilbert-imdb-sentiment/commit/e5934346215eb21390b2e8be1c67306553f2e5c4', commit_message='Upload tokenizer', commit_description='', oid='e5934346215eb21390b2e8be1c67306553f2e5c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ncncomplete/distilbert-imdb-sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='ncncomplete/distilbert-imdb-sentiment'), pr_revision=None, pr_num=None)